# Rule Set 3

![Rule Set 3](https://proto-bio.github.io/proto-assets/images/tool/rs3/hero.png)

This notebook demonstrates `run_rs3_score`, which predicts SpCas9 guide activity from a 30-nucleotide target context with the published [Rule Set 3](https://github.com/gpp-rnd/rs3) sequence model. The score is a raw mean-centered activity estimate, not a calibrated probability, so it is useful for ranking candidates within the model's screening domain.

In [ ]:
from proto_tools.utils.notebook_docs import display_api_reference
from proto_tools.tools.sequence_scoring.rs3 import (
    RS3ScoreConfig,
    RS3ScoreInput,
    run_rs3_score,
)

## API Reference

In [ ]:
display_api_reference("rs3-score", "input", "run_rs3_score")

In [ ]:
display_api_reference("rs3-score", "config", "run_rs3_score")

In [ ]:
display_api_reference("rs3-score", "output", "run_rs3_score")

## Score Target Contexts

Each context is exactly 30 nt: a 4-nt upstream flank, the 20-nt protospacer, the 3-nt NGG PAM, and a 3-nt downstream flank. Scoring an isolated 20-mer is rejected, because the model was trained on the flanking bases as well.

In [ ]:
contexts = [
    "GACGAAAGCGACAACGCGTTCATCCGGGCA",
    "AGAAAACACTAGCATCCCCACCCGCGGACT",
]
result = run_rs3_score(RS3ScoreInput(sequences=contexts), RS3ScoreConfig())

for item in result.results:
    print(f"{item.protospacer} {item.pam}  {item.metrics.score:+.3f}")

## Match the tracrRNA Scaffold

Rule Set 3 was trained with both common tracrRNA architectures and scores them separately. Pick the one that matches the construct you will build: the ranking of two guides can differ between scaffolds.

In [ ]:
for tracr in ("Hsu2013", "Chen2013"):
    scored = run_rs3_score(RS3ScoreInput(sequences=contexts), RS3ScoreConfig(tracr_rna=tracr))
    print(tracr, [round(item.metrics.score, 3) for item in scored.results])

## Rank Enumerated Guides

`crispr-guides-enumerate` returns a `context_sequence` for each candidate, which is this model's 30-nt window under the default flanks, so the two tools chain directly. A candidate near an interval edge keeps a shorter context, and flanks carry any non-ACGT reference bases, so filter to full-length ACGT contexts before scoring.

In [ ]:
from proto_tools.tools.gene_annotation.crispr_guides import (
    CRISPRGuideEnumerationConfig,
    CRISPRGuideEnumerationInput,
    CRISPRGuideRegion,
    run_crispr_guides_enumerate,
)

guides = run_crispr_guides_enumerate(
    CRISPRGuideEnumerationInput(
        regions=CRISPRGuideRegion(
            sequence_id="synthetic",
            sequence="AAAAGACTGCATTGCATGACTGCATGGAAA",
        )
    ),
    CRISPRGuideEnumerationConfig(strands="plus"),
)
candidates = [
    c
    for region in guides.results
    for c in region.candidates
    if len(c.context_sequence) == 30 and set(c.context_sequence) <= set("ACGT")
]

ranked = run_rs3_score(
    RS3ScoreInput(sequences=[c.context_sequence for c in candidates]),
    RS3ScoreConfig(),
)
for candidate, scored in sorted(
    zip(candidates, ranked.results), key=lambda pair: pair[1].metrics.score, reverse=True
):
    print(f"{candidate.protospacer} {candidate.protospacer_start}-{candidate.protospacer_end} {scored.metrics.score:+.3f}")

## Export Results

JSON keeps the full nested result; CSV flattens to one row per context with its score, tracrRNA, and model version.

In [ ]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="rs3_scores", export_path=output_dir, file_format="csv")
print(f"Rule Set 3 scores exported to {output_dir}")